# CPMC with a site-basis multi-determinant trial

This notebook follows `cpmc.ipynb`. There, the fast update (Wick's theorem for the overlap ratio, plus an $O(N^2)$ update of the Green's function) is derived for a single general trial determinant. Here the trial is a multi-Slater-determinant (MSD) expansion whose determinants are **site configurations**, as in `resampled_msd_cpmc.py` and `sampled_msd_cpmc.py`:

\begin{equation*}
\ket{\psi_T} = \sum_{k=1}^{K} c_k \ket{D_k}.
\end{equation*}

Each $\ket{D_k}$ puts one electron on each spin orbital of an occupation set $S_k \subset \{0,\dots,2n_o-1\}$, $|S_k| = n_e$, with spin orbitals ordered all up then all down as in `cpmc.ipynb`. We write

\begin{equation*}
n_p(k) = \begin{cases} 1 & p \in S_k\\ 0 & p \notin S_k\end{cases}
\end{equation*}

for the occupation numbers of determinant $k$. As a Slater determinant matrix, $\psi_k$ is the $2n_o\times n_e$ matrix whose columns are the unit vectors $e_p$, $p\in S_k$. The walker $\phi$ is a UHF determinant, $\phi = \text{diag}(\phi_\uparrow, \phi_\downarrow)$.

trot's `MultiGhfTrial` handles an MSD trial by applying the machinery of `cpmc.ipynb` to every determinant separately. We show that for site configurations almost all of that machinery can be skipped: the Green's function elements the field update needs are known in advance, and each determinant can be carried through the site loop by a single number.

In [1]:
import time
import numpy as np
import scipy as sp

np.set_printoptions(precision=4, suppress=True)
np.random.seed(0)

n_o = 8
n_e = (4, 4)
K = 50                                              # determinants in the trial


def random_config(n_o, n_e):
    up = np.sort(np.random.choice(n_o, n_e[0], replace=False))
    dn = np.sort(np.random.choice(n_o, n_e[1], replace=False))
    return up, dn


def site_det_matrix(up, dn, n_o):
    # psi_k: 2n_o x n_e, unit columns at the occupied spin orbitals
    psi = np.zeros((2 * n_o, len(up) + len(dn)))
    psi[up, np.arange(len(up))] = 1
    psi[n_o + dn, len(up) + np.arange(len(dn))] = 1
    return psi


configs = [random_config(n_o, n_e) for _ in range(K)]
coeffs = np.random.randn(K)
psi_k = [site_det_matrix(up, dn, n_o) for up, dn in configs]
occ = np.array([np.r_[np.isin(np.arange(n_o), up), np.isin(np.arange(n_o), dn)]
                for up, dn in configs]).astype(float)          # occ[k, p] = n_p(k)

phi_up = np.random.randn(n_o, n_e[0])
phi_down = np.random.randn(n_o, n_e[1])
phi = sp.linalg.block_diag(phi_up, phi_down)                   # UHF walker, 2n_o x n_e

## Overlap

For one determinant, `cpmc.ipynb` gives $\bra{D_k}\phi\rangle = \det(\psi_k^\dagger\phi)$. The columns of $\psi_k$ are unit vectors, so $\psi_k^\dagger\phi = \phi[S_k,:]$ just picks the rows of $\phi$ in $S_k$. A UHF walker does not mix spins, so these rows form a block diagonal matrix and

\begin{equation*}
\bra{D_k}\phi\rangle = \det\big(\phi_\uparrow[S_k^\uparrow,:]\big)\,\det\big(\phi_\downarrow[S_k^\downarrow,:]\big) \equiv M_\uparrow(S_k^\uparrow)\,M_\downarrow(S_k^\downarrow).
\end{equation*}

$M_\sigma(S)$ is a *minor* of the walker: the determinant of the $n_{e\sigma}\times n_{e\sigma}$ matrix made of the rows $S$ of $\phi_\sigma$. We call

\begin{equation*}
w_k = c_k\, M_\uparrow(S_k^\uparrow)\, M_\downarrow(S_k^\downarrow)
\end{equation*}

the weight of determinant $k$, so that $\bra{\psi_T}\phi\rangle = \sum_k w_k$.

A minor depends on one spin string only, and many determinants share strings, so each distinct string's minor is computed once. `MultiGhfTrial` instead forms the dense $n_e\times n_e$ matrix $\psi_k^\dagger\phi$ and its determinant for every $k$.

In [2]:
minor = lambda C, S: np.linalg.det(C[S])
w = np.array([c * minor(phi_up, up) * minor(phi_down, dn) for c, (up, dn) in zip(coeffs, configs)])
overlap = sum(c * np.linalg.det(p.T.conj() @ phi) for c, p in zip(coeffs, psi_k))
np.allclose(w.sum(), overlap)

True

## Green's function of a site determinant

**Index convention.** As in `cpmc.ipynb`, spin orbitals carry a single index $p\in\{0,\dots,2n_o-1\}$, all up then all down: site $x$ with spin up is $p = x$, and site $x$ with spin down is $p = x+n_o$. We write

\begin{equation*}
x\uparrow \equiv x, \qquad x\downarrow \equiv x+n_o .
\end{equation*}

So $G^k$ is a $2n_o\times 2n_o$ matrix made of four $n_o\times n_o$ spin blocks,

\begin{equation*}
G^k = \begin{pmatrix} G^k_{\uparrow\uparrow} & G^k_{\uparrow\downarrow}\\ G^k_{\downarrow\uparrow} & G^k_{\downarrow\downarrow}\end{pmatrix},
\end{equation*}

and $G^k_{x\uparrow,\,x\downarrow} = G^k_{x,\,x+n_o}$ is an element of the off-diagonal block $G^k_{\uparrow\downarrow}$. The on-site Hubbard field at site $x$ involves exactly the pair $x\uparrow$ and $x\downarrow$.

The Green's function of determinant $k$ is, as in `cpmc.ipynb`,

\begin{equation*}
G^k_{pq} = \frac{\bra{D_k}c_p^{\dagger}c_q\ket{\phi}}{\bra{D_k}\phi\rangle} = \big(\phi\, O_k^{-1}\psi_k^\dagger\big)_{qp},\qquad O_k = \psi_k^\dagger\phi = \phi[S_k,:].
\end{equation*}

Column $p$ of $\psi_k^\dagger$ is the unit vector $e_m$ if $p$ is the $m$-th element of $S_k$, and zero if $p\notin S_k$. So **row $p$ of $G^k$ vanishes unless $p\in S_k$**. Physically, $\bra{D_k}c_p^\dagger$ is zero unless spin orbital $p$ is occupied in $D_k$. Two facts follow.

1. **The diagonal is the occupation.** For $p\in S_k$, row $p$ of $\phi$ is row $m$ of $O_k$, so $G^k_{pp} = (O_kO_k^{-1})_{mm} = 1$. For $p\notin S_k$ the whole row is zero. Hence

\begin{equation*}
G^k_{pp} = n_p(k)
\end{equation*}

for every walker $\phi$. Physically, $\ket{D_k}$ is an eigenstate of every $n_p = c_p^\dagger c_p$, so $\bra{D_k}n_p = n_p(k)\bra{D_k}$.

2. **Up and down do not mix: the off-diagonal spin blocks vanish.** Both $\phi$ and $\psi_k$ are block diagonal in spin. Their first $n_{e\uparrow}$ columns (the up electrons) are nonzero only in the up rows $0,\dots,n_o-1$, and the remaining columns only in the down rows $n_o,\dots,2n_o-1$. Therefore

\begin{equation*}
O_k = \psi_k^\dagger\phi = \begin{pmatrix} \phi_\uparrow[S_k^\uparrow,:] & 0\\ 0 & \phi_\downarrow[S_k^\downarrow,:]\end{pmatrix}
\end{equation*}

is block diagonal, so is $O_k^{-1}$, and so is the product $\phi\,O_k^{-1}\psi_k^\dagger$:

\begin{equation*}
G^k = \begin{pmatrix} G^k_{\uparrow\uparrow} & 0\\ 0 & G^k_{\downarrow\downarrow}\end{pmatrix}.
\end{equation*}

In particular, for the two spin orbitals of site $x$,

\begin{equation*}
G^k_{x\uparrow,\,x\downarrow} = G^k_{x,\,x+n_o} = 0,\qquad G^k_{x\downarrow,\,x\uparrow} = G^k_{x+n_o,\,x} = 0.
\end{equation*}

Physically, $G^k_{x\uparrow,x\downarrow} = \bra{D_k}c^\dagger_{x\uparrow}c_{x\downarrow}\ket{\phi}/\bra{D_k}\phi\rangle$ is a spin-flip amplitude. $c^\dagger_{x\uparrow}c_{x\downarrow}$ turns a down electron into an up one, changing $(N_\uparrow, N_\downarrow)$ by $(+1,-1)$, and a state with the wrong particle numbers has zero overlap with $\bra{D_k}$.

Fact 2 is not special to site determinants: it holds for any spin-conserving trial with a UHF walker, and it is why the Wick ratio below factorises into an up factor times a down factor. Fact 1 is the one special to the site basis, and it is what removes the need for $G^k$ altogether.

Below, the nonzero pattern of $G^k$ for the first determinant (1 = nonzero): only the rows of occupied spin orbitals are filled, and only inside the two diagonal spin blocks.

Putting the pieces together, $G^k$ is **not** the identity: its structure is fixed by which spin orbitals $D_k$ occupies. With $m(p)$ the position of $p$ in $S_k$,

\begin{equation*}
G^k_{pq} = \begin{cases}
0 & p\notin S_k,\\[2pt]
\delta_{pq} & p\in S_k,\ q\in S_k,\\[2pt]
\big(\phi\,O_k^{-1}\big)_{q,\,m(p)} & p\in S_k,\ q\notin S_k.
\end{cases}
\end{equation*}

- **$p$ unoccupied.** The row is zero, because $\bra{D_k}c_p^\dagger = 0$.
- **$p$ and $q$ both occupied.** Row $q$ of $\phi$ is row $m(q)$ of $O_k$, so $G^k_{pq} = (O_kO_k^{-1})_{m(q)\,m(p)} = \delta_{pq}$. Physically, for $p\neq q$, $\bra{D_k}c_p^\dagger c_q = \big(c_q^\dagger c_p\ket{D_k}\big)^\dagger = 0$, because $c_q^\dagger$ acts on an orbital that is already occupied. This block does not depend on the walker; its diagonal is fact 1.
- **$p$ occupied, $q$ empty.** This block is the only one that depends on the walker. It holds the amplitudes for moving an electron between an orbital $D_k$ occupies and one it leaves empty, and it is what the hopping energy needs (see the local-energy section).

In [3]:
def green(psi, phi):
    # cpmc.ipynb's G(phi)_{pq} for one trial determinant psi
    return (phi @ np.linalg.inv(psi.T.conj() @ phi) @ psi.T.conj()).T


G = [green(p, phi) for p in psi_k]
print(G[0])
print("occupations n_p(0), p = 0 ... 2n_o-1:", occ[0].astype(int))
print((np.abs(G[0]) > 1e-12).astype(int))

diag_is_occupation = all(np.allclose(np.diag(g), o) for g, o in zip(G, occ))
spin_blocks_zero = all(np.allclose(g[:n_o, n_o:], 0) and np.allclose(g[n_o:, :n_o], 0) for g in G)
diag_is_occupation, spin_blocks_zero

[[ 0.      0.      0.      0.      0.      0.      0.      0.      0.
   0.      0.      0.      0.      0.      0.      0.    ]
 [ 0.2845  1.      0.      1.6772  1.0029  0.2278  0.     -0.      0.
   0.      0.      0.      0.      0.      0.      0.    ]
 [ 0.6533 -0.      1.     -0.9098 -0.2218 -0.0809  0.      0.      0.
   0.      0.      0.      0.      0.      0.      0.    ]
 [ 0.      0.      0.      0.      0.      0.      0.      0.      0.
   0.      0.      0.      0.      0.      0.      0.    ]
 [ 0.      0.      0.      0.      0.      0.      0.      0.      0.
   0.      0.      0.      0.      0.      0.      0.    ]
 [ 0.      0.      0.      0.      0.      0.      0.      0.      0.
   0.      0.      0.      0.      0.      0.      0.    ]
 [ 0.5872  0.      0.     -1.9295 -0.6503 -0.2434  1.      0.      0.
   0.      0.      0.      0.      0.      0.      0.    ]
 [-1.29    0.      0.      1.4805  0.3093  0.2699  0.      1.      0.
   0.      0.      0.      

(True, True)

## Field update, step by step

The interaction propagator is applied one site at a time. For the on-site Hubbard interaction at site $x$, `cpmc.ipynb`'s Hubbard-Stratonovich transformation reads

\begin{equation*}
e^{-\Delta\tau U n_{x\uparrow}n_{x\downarrow}} = \sum_{s=\pm1}\frac{1}{2}\hat B(s),\qquad \hat B(s) = e^{\lambda_+(s)\,n_{x\uparrow}}\,e^{\lambda_-(s)\,n_{x\downarrow}},
\end{equation*}

\begin{equation*}
\lambda_+(s) = \gamma s - \frac{\Delta\tau U}{2},\qquad \lambda_-(s) = -\gamma s - \frac{\Delta\tau U}{2},\qquad \cosh\gamma = e^{\Delta\tau U/2}.
\end{equation*}

On a Slater determinant, $\hat B(s)$ multiplies row $x\uparrow$ of $\phi$ by $e^{\lambda_+(s)}$ and row $x\downarrow$ by $e^{\lambda_-(s)}$ (`cpmc.ipynb`). We write $c_\pm(s) = e^{\lambda_\pm(s)}-1$ and, as before, $i = x\uparrow = x$ and $j = x\downarrow = x+n_o$.

A field update at site $x$ has four steps:

1. For each $s=\pm1$, the overlap ratio of every trial determinant, $R_k(s) = \bra{D_k}\hat B(s)\ket{\phi}/\bra{D_k}\phi\rangle$.
2. The ratio of the whole trial, $R(s) = \bra{\psi_T}\hat B(s)\ket{\phi}/\bra{\psi_T}\phi\rangle$.
3. Pick $s$ by importance sampling with $R(s)$, and reweight the walker.
4. Apply $\hat B(s)$ to $\phi$, and bring up to date whatever step 1 will need at the next site.

The site loop repeats this for $x = 0,\dots,n_o-1$. `MultiGhfTrial` and `resampled_msd_cpmc.py` do the same four steps. They differ only in what steps 1 and 4 compute.

In [4]:
dt, U = 0.01, 4.0                                  # time step and Hubbard U
gamma = np.arccosh(np.exp(dt * U / 2))


def lam(s):
    # (lambda_+(s), lambda_-(s)) for the field s = +1 or -1 at one site
    return gamma * s - dt * U / 2, -gamma * s - dt * U / 2


for s in (+1, -1):
    print(f"s = {s:+d}:  e^lambda_+ = {np.exp(lam(s)[0]):.6f},  e^lambda_- = {np.exp(lam(s)[1]):.6f}")

s = +1:  e^lambda_+ = 1.198017,  e^lambda_- = 0.801983
s = -1:  e^lambda_+ = 0.801983,  e^lambda_- = 1.198017


### Step 1: the ratio of one determinant, $R_k(s)$

**`MultiGhfTrial`** evaluates it with `cpmc.ipynb`'s Wick formula, from the stored Green's function of determinant $k$:

\begin{equation*}
R_k(s) = \big(1 + c_+G^k_{ii}\big)\big(1 + c_-G^k_{jj}\big) - c_+c_-\,G^k_{ij}G^k_{ji}.
\end{equation*}

**Site basis.** $\ket{D_k}$ is an eigenstate of every occupation number, so $n_p$ acting to the left on the bra just gives a number: $\bra{D_k}n_p = n_p(k)\bra{D_k}$. Expand $\hat B(s) = 1 + c_+n_i + c_-n_j + c_+c_-n_in_j$ (`cpmc.ipynb`) and let it act on the bra:

\begin{equation*}
\bra{D_k}\hat B(s) = \big(1 + c_+n_i(k)\big)\big(1 + c_-n_j(k)\big)\bra{D_k}
\quad\Longrightarrow\quad
R_k(s) = \big(1 + c_+(s)\,n_{x\uparrow}(k)\big)\big(1 + c_-(s)\,n_{x\downarrow}(k)\big).
\end{equation*}

Neither a Green's function nor the walker enters. $R_k(s)$ takes one of only four values, depending on what $D_k$ has at site $x$:

| site $x$ in $D_k$ | $R_k(s)$ |
|---|---|
| empty | $1$ |
| $\uparrow$ only | $e^{\lambda_+(s)}$ |
| $\downarrow$ only | $e^{\lambda_-(s)}$ |
| doubly occupied | $e^{\lambda_+(s)+\lambda_-(s)}$ |

The Wick formula gives the same result, because $G^k_{ii} = n_i(k)$, $G^k_{jj} = n_j(k)$ and $G^k_{ij} = G^k_{ji} = 0$ (previous section).

In [5]:
x = 3
i, j = x, x + n_o
checks = []
for s in (+1, -1):
    lp, ln = lam(s)
    cp, cn = np.exp(lp) - 1, np.exp(ln) - 1
    phi_s = phi.copy()
    phi_s[i] *= np.exp(lp)
    phi_s[j] *= np.exp(ln)
    R_direct = np.array([np.linalg.det(p.T.conj() @ phi_s) / np.linalg.det(p.T.conj() @ phi) for p in psi_k])
    R_wick = np.array([(1 + cp * g[i, i]) * (1 + cn * g[j, j]) - cp * cn * g[i, j] * g[j, i]
                       for g in G])                                            # MultiGhfTrial
    R_occ = (1 + cp * occ[:, i]) * (1 + cn * occ[:, j])                        # site basis
    checks += [np.allclose(R_direct, R_wick), np.allclose(R_direct, R_occ)]
print("the distinct values R_k(-1) takes over all determinants:", np.unique(np.round(R_occ, 12)))
all(checks)

the distinct values R_k(-1) takes over all determinants: [0.802  0.9608 1.     1.198 ]


True

### Step 2: the ratio of the whole trial, $R(s)$

The trial is a sum of determinants, so its ratio is the weighted average of theirs, with the weights $w_k = c_k\bra{D_k}\phi\rangle$ from the overlap section:

\begin{equation*}
R(s) = \frac{\sum_k c_k\bra{D_k}\hat B(s)\ket{\phi}}{\sum_k c_k\bra{D_k}\phi\rangle} = \frac{\sum_k w_k\,R_k(s)}{\sum_k w_k}.
\end{equation*}

Both codes compute exactly this in `calc_overlap_ratio`. In the site basis it has a transparent form. Sort the determinants by what they have at site $x$, and add up the weights in each of the four groups,

\begin{equation*}
T_{ab} = \sum_{k:\ n_{x\uparrow}(k)=a,\ n_{x\downarrow}(k)=b} w_k,\qquad a,b\in\{0,1\}.
\end{equation*}

Then

\begin{equation*}
R(s) = \frac{T_{00} + e^{\lambda_+(s)}\,T_{10} + e^{\lambda_-(s)}\,T_{01} + e^{\lambda_+(s)+\lambda_-(s)}\,T_{11}}{T_{00}+T_{10}+T_{01}+T_{11}}.
\end{equation*}

The four sums $T_{ab}$ do not depend on $s$, so one pass over the determinants gives the ratio for both field values.

In [6]:
T = {(a, b): w[(occ[:, i] == a) & (occ[:, j] == b)].sum() for a in (0, 1) for b in (0, 1)}
checks = []
for s in (+1, -1):
    lp, ln = lam(s)
    cp, cn = np.exp(lp) - 1, np.exp(ln) - 1
    phi_s = phi.copy()
    phi_s[i] *= np.exp(lp)
    phi_s[j] *= np.exp(ln)
    R_direct = sum(c * np.linalg.det(p.T.conj() @ phi_s) for c, p in zip(coeffs, psi_k)) / overlap
    R_weights = (w * (1 + cp * occ[:, i]) * (1 + cn * occ[:, j])).sum() / w.sum()
    R_groups = (T[0, 0] + np.exp(lp) * T[1, 0] + np.exp(ln) * T[0, 1] + np.exp(lp + ln) * T[1, 1]) / sum(T.values())
    checks += [np.allclose(R_direct, R_weights), np.allclose(R_direct, R_groups)]
all(checks)

True

### Step 3: choose the field

This step is the same for any trial (`cpmc.ipynb`, and trot's `cpmc_step`). The walker $\sum_s\frac12\hat B(s)\ket{\phi}$ is importance sampled: $s$ is drawn with probability

\begin{equation*}
P(s) = \frac{\tfrac12\max\big(0, R(s)\big)}{\mathcal N},\qquad \mathcal N = \sum_{s=\pm1}\tfrac12\max\big(0, R(s)\big),
\end{equation*}

and the walker's weight is multiplied by $\mathcal N$. A negative ratio means $\hat B(s)$ would move the walker across the trial's node, so the constraint excludes that field. If both fields are excluded, the walker's weight becomes zero.

### Step 4: update, and what does *not* need updating

Once the field $s$ is chosen:

* **The walker.** Rows $x\uparrow$ and $x\downarrow$ of $\phi$ are multiplied by $e^{\lambda_+(s)}$ and $e^{\lambda_-(s)}$.
* **The trial's overlap.** $\bra{\psi_T}\phi\rangle \to R(s)\,\bra{\psi_T}\phi\rangle$.
* **The determinant weights** (`update_green` in `resampled_msd_cpmc.py`). $w_k \to R_k(s)\,w_k$, since $w_k$ is $c_k$ times the overlap of determinant $k$.

That is the whole update in the site basis. Step 1 at the next site $x'$ reads only $n_{x'\uparrow}(k)$ and $n_{x'\downarrow}(k)$, which are properties of $D_k$ and never change.

`MultiGhfTrial` has one more item. Step 1 at the next site reads $G^k$, so every determinant's Green's function must be brought up to date, $G^k \to G^k(\hat B(s)\phi)$. `cpmc.ipynb` shows how to do this in $O(N^2)$ with a rank-2 update. For a site determinant, that update recomputes numbers that are already known: the diagonal comes back as $n_p(k)$, and the $x\uparrow,x\downarrow$ elements stay zero. The elements it does change, in the occupied-empty block, are never read by the site loop. The check below applies `cpmc.ipynb`'s update to every determinant.

In [7]:
def green_update(g, i, j, cons_p, cons_n):
    # cpmc.ipynb's O(N^2) Green's function update for one determinant
    ratio = (1 + cons_p * g[i, i]) * (1 + cons_n * g[j, j]) - cons_p * cons_n * g[i, j] * g[j, i]
    sg_i = g[i].copy()
    sg_i[i] -= 1
    sg_j = g[j].copy()
    sg_j[j] -= 1
    return (
        g
        + (cons_p / ratio) * np.outer(g[:, i], cons_n * (g[i, j] * sg_j - g[j, j] * sg_i) - sg_i)
        + (cons_n / ratio) * np.outer(g[:, j], cons_p * (g[j, i] * sg_i - g[i, i] * sg_j) - sg_j)
    )


s = +1
lambda_p, lambda_n = lam(s)
cons_p, cons_n = np.exp(lambda_p) - 1, np.exp(lambda_n) - 1
phi_p = phi.copy()
phi_p[i] *= np.exp(lambda_p)
phi_p[j] *= np.exp(lambda_n)
G_p = [green_update(g, i, j, cons_p, cons_n) for g in G]
update_is_exact = all(np.allclose(gp, green(p, phi_p)) for gp, p in zip(G_p, psi_k))
diag_unchanged = np.allclose([np.diag(gp) for gp in G_p], occ)
largest_change = max(np.abs(gp - g).max() for gp, g in zip(G_p, G))
update_is_exact, diag_unchanged, largest_change

(True, True, np.float64(363.02536334468846))

### The whole site loop

Here is one sweep over all sites, the four steps at each, done both ways with the same random numbers. `sweep_multighf` carries every $G^k$ through the loop, as `MultiGhfTrial` does. `sweep_site_basis` carries only the weights $w_k$, as `resampled_msd_cpmc.py` does. They choose the same fields and end with the same weights and the same walker weight factor. The weights also agree with overlaps recomputed from scratch at the end.

In [8]:
def choose(R_plus, R_minus, u):
    # step 3: importance-sample s; return (s, walker weight factor N)
    p_plus, p_minus = 0.5 * max(0.0, R_plus), 0.5 * max(0.0, R_minus)
    norm = p_plus + p_minus
    return (+1 if u < p_plus / norm else -1), norm


def sweep_site_basis(phi, w, us):
    phi, w, factor, fields = phi.copy(), w.copy(), 1.0, []
    for x in range(n_o):
        i, j = x, x + n_o
        Rk = {s: (1 + (np.exp(lam(s)[0]) - 1) * occ[:, i]) * (1 + (np.exp(lam(s)[1]) - 1) * occ[:, j])
              for s in (+1, -1)}                                          # step 1, from occupations
        R = {s: (w * Rk[s]).sum() / w.sum() for s in (+1, -1)}           # step 2
        s, norm = choose(R[+1], R[-1], us[x])                            # step 3
        phi[i] *= np.exp(lam(s)[0])                                      # step 4: walker
        phi[j] *= np.exp(lam(s)[1])
        w = w * Rk[s]                                                    # step 4: weights, and done
        factor, fields = factor * norm, fields + [s]
    return phi, w, factor, fields


def sweep_multighf(phi, G, w, us):
    phi, G, w, factor, fields = phi.copy(), [g.copy() for g in G], w.copy(), 1.0, []
    for x in range(n_o):
        i, j = x, x + n_o
        Rk = {}
        for s in (+1, -1):                                               # step 1, from G^k
            cp, cn = np.exp(lam(s)[0]) - 1, np.exp(lam(s)[1]) - 1
            Rk[s] = np.array([(1 + cp * g[i, i]) * (1 + cn * g[j, j]) - cp * cn * g[i, j] * g[j, i]
                              for g in G])
        R = {s: (w * Rk[s]).sum() / w.sum() for s in (+1, -1)}           # step 2
        s, norm = choose(R[+1], R[-1], us[x])                            # step 3
        cp, cn = np.exp(lam(s)[0]) - 1, np.exp(lam(s)[1]) - 1
        phi[i] *= np.exp(lam(s)[0])                                      # step 4: walker
        phi[j] *= np.exp(lam(s)[1])
        w = w * Rk[s]                                                    # step 4: weights
        G = [green_update(g, i, j, cp, cn) for g in G]                   # step 4: every G^k, O(N^2) each
        factor, fields = factor * norm, fields + [s]
    return phi, w, factor, fields


us = np.random.rand(n_o)
phi_a, w_a, f_a, s_a = sweep_site_basis(phi, w, us)
phi_b, w_b, f_b, s_b = sweep_multighf(phi, G, w, us)
w_recomputed = np.array([c * np.linalg.det(p.T.conj() @ phi_a) for c, p in zip(coeffs, psi_k)])
print("fields chosen:", s_a)
s_a == s_b, np.allclose(w_a, w_b), np.isclose(f_a, f_b), np.allclose(w_a, w_recomputed)

fields chosen: [1, -1, 1, 1, 1, -1, 1, -1]


(True, True, np.True_, True)

## Local energy

For $H = \sum_{pq} h_{pq}\, c_p^{\dagger}c_q + U\sum_x n_{x\uparrow}n_{x\downarrow}$ (with $h$ block diagonal in spin) the local energy of the trial is again a weighted average,

\begin{equation*}
E_L(\phi) = \frac{\bra{\psi_T}H\ket{\phi}}{\bra{\psi_T}\phi\rangle} = \frac{\sum_k w_k\,\varepsilon_k}{\sum_k w_k},
\end{equation*}

\begin{equation*}
\varepsilon_k = \frac{\bra{D_k}H\ket{\phi}}{\bra{D_k}\phi\rangle} = \sum_{pq} h_{pq}G^k_{pq} + U\sum_x\big(G^k_{x\uparrow x\uparrow}G^k_{x\downarrow x\downarrow} - G^k_{x\uparrow x\downarrow}G^k_{x\downarrow x\uparrow}\big).
\end{equation*}

This is `MultiGhfTrial`'s energy: `_energy_from_full_green` on the full $2n_o\times 2n_o$ $G^k$ of every determinant.

For a site determinant, the interaction term is $U d_k$, where $d_k = \sum_x n_{x\uparrow}(k)\,n_{x\downarrow}(k)$ is the number of doubly occupied sites. Only the rows $p\in S_k$ of $G^k$ are nonzero, and per spin those rows, transposed, form the $n_o\times n_{e\sigma}$ matrix $\phi_\sigma(\phi_\sigma[S_\sigma,:])^{-1}$: its column $m$ is row $p_m$ of $G^k$, where $p_m$ is the $m$-th element of $S_k^\sigma$. So

\begin{equation*}
\varepsilon_k = \sum_{\sigma}\text{tr}\Big(h[S_k^\sigma,:]\;\phi_\sigma\big(\phi_\sigma[S_k^\sigma,:]\big)^{-1}\Big) + U d_k,
\end{equation*}

where $h[S,:]$ are the rows $S$ of the one-body matrix. This is `energy_kernel` in `resampled_msd_cpmc.py`. It needs one $n_{e\sigma}\times n_{e\sigma}$ solve per distinct string and spin, done once per block when the energy is measured, and it is never carried through the site loop. Of all the elements of $G^k$, this is the only place the off-diagonal ones are needed.

In [9]:
h = np.zeros((n_o, n_o))
h[np.arange(n_o - 1), np.arange(1, n_o)] = h[np.arange(1, n_o), np.arange(n_o - 1)] = -1.0
U = 4.0


def eps_full_green(g):
    # MultiGhfTrial: the energy from the full 2n_o x 2n_o Green's function
    e1 = np.sum(g[:n_o, :n_o] * h) + np.sum(g[n_o:, n_o:] * h)
    e2 = U * (np.sum(np.diag(g[:n_o, :n_o]) * np.diag(g[n_o:, n_o:]))
              - np.sum(np.diag(g[:n_o, n_o:]) * np.diag(g[n_o:, :n_o])))
    return e1 + e2


def eps_site(up, dn):
    # resampled_msd_cpmc.py: tr(h[S,:] phi phi[S,:]^-1) per spin, plus U times doubly occupied sites
    k1 = lambda C, S: np.trace(h[S] @ C @ np.linalg.inv(C[S]))
    return k1(phi_up, up) + k1(phi_down, dn) + U * len(np.intersect1d(up, dn))


E_multighf = (w * np.array([eps_full_green(g) for g in G])).sum() / w.sum()
E_site = (w * np.array([eps_site(up, dn) for up, dn in configs])).sum() / w.sum()
np.allclose(E_multighf, E_site)

True

## Minors from reference tables

With `TABLE_MSD` on, the minors themselves are computed more cheaply than by one LU per string. Pick a reference string $R$ and form, once per walker and spin,

\begin{equation*}
T = \phi_\sigma\,\big(\phi_\sigma[R,:]\big)^{-1}\qquad (n_o\times n_{e\sigma}).
\end{equation*}

A string $S$ that differs from $R$ by $k$ orbitals, with created sites $\text{cre} = S\setminus R$ and annihilated sites $\text{ann} = R\setminus S$ (both sorted), then has

\begin{equation*}
M_\sigma(S) = (-1)^{\sum I + \sum J}\,\det\big(T[\text{cre}, J]\big)\,M_\sigma(R).
\end{equation*}

Here $I$ are the positions of $\text{cre}$ in $S$, and $J$ are the positions of $\text{ann}$ in $R$. Each string then costs a $k\times k$ determinant (closed form for $k\le 3$) instead of an $n_{e\sigma}\times n_{e\sigma}$ one. With a few well-chosen references, the typical $k$ is 2 to 3 at $n_{e\sigma} = 8$ to $12$.

In [10]:
R = np.array([0, 1, 2, 3])
S = np.array([0, 2, 5, 7])                                   # k = 2 away from R
T = phi_up @ np.linalg.inv(phi_up[R])
cre, ann = np.setdiff1d(S, R), np.setdiff1d(R, S)
I, J = np.searchsorted(S, cre), np.searchsorted(R, ann)
sign = (-1) ** (I.sum() + J.sum())
np.allclose(minor(phi_up, S), sign * np.linalg.det(T[np.ix_(cre, J)]) * minor(phi_up, R))

True

## What `MultiGhfTrial` does, and what it costs

`MultiGhfTrial` is written for general GHF determinants: each $\psi_k$ is a dense $2n_o\times n_e$ matrix, and the machinery of `cpmc.ipynb` is applied to every determinant. Per walker, per determinant:

| | `MultiGhfTrial` | site-basis MSD (`resampled_msd_cpmc.py`) |
|---|---|---|
| overlap | dense $\psi_k^\dagger\phi$ and an $n_e\times n_e$ determinant | two minors $M_\uparrow, M_\downarrow$, shared by all determinants with the same string ($k\times k$ with tables) |
| stored during the site loop | $G^k$: $(2n_o)^2$ numbers | $w_k$: one number |
| ratio at one site | Wick formula from $G^k$ | $(1+c_+n_{x\uparrow})(1+c_-n_{x\downarrow})$ from the occupations |
| update at one site | $O((2n_o)^2)$ rank-2 update of $G^k$ | $w_k\to w_kR_k$: one multiply |
| energy, once per block | builds $G^k$: $n_e\times n_e$ solve with $2n_o$ right-hand sides, plus products | per distinct string and spin: $n_{e\sigma}\times n_{e\sigma}$ solve with $n_o$ right-hand sides |

A sweep visits all $n_o$ sites. `MultiGhfTrial` therefore spends $\sim n_o(2n_o)^2$ operations per determinant per sweep keeping $G^k$ current, against $n_o$ multiplies here. Nothing is approximated: every number that is skipped is either known in advance ($G^k_{pp}$, $G^k_{x\uparrow x\downarrow}$) or never read in the site loop (the rest of $G^k$).

`MultiGhfTrial` is the right tool when the determinants are general, for example symmetry-projected GHF states. Then $G^k$ is dense and genuinely changes at every update, and there are usually few determinants. A site-configuration trial is the opposite case: thousands of determinants whose $G^k$ diagonal never changes.

One site update at a production size, $n_o = 16$ with the 4365 determinants of a 5000-draw trial, for one walker:

In [11]:
def green_update_batch(g, i, j, cons_p, cons_n):
    # green_update for a stack of determinants, g: (K, 2n_o, 2n_o)
    gii, gjj, gij, gji = g[:, i, i], g[:, j, j], g[:, i, j], g[:, j, i]
    ratio = (1 + cons_p * gii) * (1 + cons_n * gjj) - cons_p * cons_n * gij * gji
    sg_i = g[:, i, :].copy()
    sg_i[:, i] -= 1
    sg_j = g[:, j, :].copy()
    sg_j[:, j] -= 1
    a = cons_n * (gij[:, None] * sg_j - gjj[:, None] * sg_i) - sg_i
    b = cons_p * (gji[:, None] * sg_i - gii[:, None] * sg_j) - sg_j
    return (g + (cons_p / ratio)[:, None, None] * g[:, :, i, None] * a[:, None, :]
              + (cons_n / ratio)[:, None, None] * g[:, :, j, None] * b[:, None, :])


i, j = 3, 3 + n_o
batched_is_same = np.allclose(green_update_batch(np.array(G), i, j, cons_p, cons_n),
                              [green_update(g, i, j, cons_p, cons_n) for g in G])

n_o_big, K_big = 16, 4365
G_big = np.random.randn(K_big, 2 * n_o_big, 2 * n_o_big)          # stand-in Green's functions
w_big = np.random.randn(K_big)
occ_big = (np.random.rand(K_big, 2 * n_o_big) < 0.5).astype(float)
i, j = 3, 3 + n_o_big


def best_of(f, n=20):
    ts = []
    for _ in range(n):
        t0 = time.perf_counter()
        f()
        ts.append(time.perf_counter() - t0)
    return min(ts)


t_multighf = best_of(lambda: green_update_batch(G_big, i, j, cons_p, cons_n))
t_site = best_of(lambda: w_big * (1 + cons_p * occ_big[:, i]) * (1 + cons_n * occ_big[:, j]))
print(f"batched update matches the per-determinant one: {batched_is_same}")
print(f"one site update, one walker: MultiGhfTrial-style {t_multighf * 1e3:.2f} ms, "
      f"site-basis {t_site * 1e6:.1f} us  ({t_multighf / t_site:.0f}x)")
print(f"stored per walker: {G_big.nbytes / 1e6:.0f} MB of Green's functions (real) "
      f"against {w_big.nbytes / 1e3:.0f} kB of weights; x200 walkers: {200 * G_big.nbytes / 1e9:.1f} GB "
      f"real, {400 * G_big.nbytes / 1e9:.1f} GB complex128 as MultiGhfTrial stores it")

batched update matches the per-determinant one: True
one site update, one walker: MultiGhfTrial-style 10.08 ms, site-basis 13.5 us  (747x)
stored per walker: 36 MB of Green's functions (real) against 35 kB of weights; x200 walkers: 7.2 GB real, 14.3 GB complex128 as MultiGhfTrial stores it


In trot itself, on the same 4365-determinant trial (L=16, 5000 draws, 32 walkers), one full block of 20 fast-update steps plus measurement takes:

| trial ops | s / block | block energy |
|---|---|---|
| `MultiGhfTrial`, complex128 | 89.6 | $-8.795307$ |
| `MultiGhfTrial`, complex64 (trot's default) | 65.7 | $-8.795308$ |
| site-basis MSD (`resampled_msd_cpmc.py`) | 4.98 | $-8.795307$ |

Both give the same energy. The gap is smaller than in the site-update timing above, because the minors, which both approaches need, dominate the site-basis block. At 200 walkers `MultiGhfTrial` would also store 14 GB of Green's functions, against a few MB of weights here.